# Prepare Training Data for Four Supervised Conditions

Prepares datasets from corpus.db for fine-tuning under four conditions:

1. **Natural-text adaptation**: input = prior line/lines in the same poem or start; output = **next line** surface text.
2. **Meter-only**: input = line text; output = stress pattern or meter label
3. **Rhyme-only**: input = line text; output = rhyme key  or rhyme class
4. **Combined**: **same input as natural_text**; output = bundled stress|meter_type|rhyme|end|caesura for that **next** line (plus next_line field for strict-eval tooling).

In [1]:
import json
import sqlite3
from pathlib import Path
from typing import Iterator

ROOT = Path("../").resolve()  # project root
DB_PATH = ROOT / "output" / "corpus.db"
SPLITS_DIR = ROOT / "evaluation" / "splits"
OUT_DIR = ROOT / "output" / "training_data"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert DB_PATH.exists(), f"Run python scripts/export_sqlite.py first. Missing: {DB_PATH}"

#### Load splits and corpus

In [2]:
def load_split(name: str) -> list:
    with open(SPLITS_DIR / f"{name}.json", encoding="utf-8") as f:
        return json.load(f)

def iter_lines(conn: sqlite3.Connection, poem_ids: list) -> Iterator[dict]:
    placeholders = ",".join("?" * len(poem_ids))
    rows = conn.execute(
        f"""SELECT poem_id, stanza_index, line_index, normalized, stress, meter_type,
                  rhyme_group, rhyme_word, end_stopped, caesura, phonology
           FROM lines WHERE poem_id IN ({placeholders}) ORDER BY poem_id, stanza_index, line_index""",
        poem_ids,
    ).fetchall()
    for r in rows:
        yield {
            "poem_id": r[0], "stanza_index": r[1], "line_index": r[2],
            "normalized": r[3], "stress": r[4], "meter_type": r[5],
            "rhyme_group": r[6], "rhyme_word": r[7],
            "end_stopped": r[8], "caesura": r[9], "phonology": r[10],
        }

train_ids = load_split("train")
dev_ids = load_split("dev")
test_ids = load_split("test")
print(f"train={len(train_ids)}, dev={len(dev_ids)}, test={len(test_ids)}")

train=2380, dev=510, test=511


## Condition 1: Natural-text adaptation

Input = previous line(s) **within the same poem** (or [start] for the first line); output = the **next line** (continuation). Context never crosses poem boundaries.

In [3]:
def continuation_context_input(
    lines: list, i: int, context_lines: int, poem_id
) -> str:
    """Prior lines from the same poem only (chronological), capped at context_lines."""
    gathered = []
    j = i - 1
    while j >= 0 and len(gathered) < context_lines:
        if lines[j]["poem_id"] != poem_id:
            break
        t = (lines[j].get("normalized") or "").strip()
        if t:
            gathered.append(t)
        j -= 1
    gathered.reverse()
    return " | ".join(gathered) if gathered else "[start]"


def build_natural_text_pairs(conn: sqlite3.Connection, poem_ids: list, context_lines: int = 1) -> list:
    out = []
    lines = list(iter_lines(conn, poem_ids))
    i = 0
    while i < len(lines):
        line = lines[i]
        text = (line.get("normalized") or "").strip()
        if not text:
            i += 1
            continue
        inp = continuation_context_input(lines, i, context_lines, line["poem_id"])
        out.append({"input": inp, "target": text})
        i += 1
    return out

conn = sqlite3.connect(DB_PATH)
nat_train = build_natural_text_pairs(conn, train_ids)
nat_dev = build_natural_text_pairs(conn, dev_ids)
nat_test = build_natural_text_pairs(conn, test_ids)
conn.close()
print(f"Natural-text: train={len(nat_train)}, dev={len(nat_dev)}, test={len(nat_test)}")

Natural-text: train=206956, dev=42831, test=43535


#### Condition 2: Meter-only structured supervision

Input = line text; Output = stress pattern +-+-+-+-+-+ or meter_type label.
According to de la Rosa, research should use stress as binary string; one wrong syllable → whole line wrong.

In [4]:
def stress_to_plus_minus(s: str) -> str:
    """Convert corpus stress to +/- format. Accepts 0/1 (lexical) or +/- (Prosodic)."""
    if not s or not isinstance(s, str):
        return ""
    s = s.strip()
    if len(s) < 5:  # Skip degenerate: "1", "11", etc.
        return ""
    if all(c in "+-" for c in s):
        return s  # Already Prosodic format
    if all(c in "01" for c in s):
        return "".join("+" if c == "1" else "-" for c in s)
    return ""


def build_meter_pairs(conn: sqlite3.Connection, poem_ids: list) -> list:
    """Only include lines with valid stress pattern (5+ syllables), output as +/-."""
    out = []
    for line in iter_lines(conn, poem_ids):
        text = (line.get("normalized") or "").strip()
        stress = line.get("stress") or ""
        if not text:
            continue
        target = stress_to_plus_minus(stress)
        if not target:
            continue  # Skip invalid/short stress
        out.append({"input": text, "target": target})
    return out


conn = sqlite3.connect(DB_PATH)
meter_train = build_meter_pairs(conn, train_ids)
meter_dev = build_meter_pairs(conn, dev_ids)
meter_test = build_meter_pairs(conn, test_ids)
conn.close()
print(f"Meter-only: train={len(meter_train)}, dev={len(meter_dev)}, test={len(meter_test)}")
if meter_train:
    ex = meter_train[0]
    print(f"  Example: input={ex['input'][:50]}... target={ex['target']}")

Meter-only: train=175767, dev=36209, test=36181
  Example: input=Tired Nature's sweet restorer, balmy Sleep!... target=+-+-+-+-+-+


#### Condition 3: Rhyme-only structured supervision

Input = line text; Output = rhyme key  or rhyme_group a, b, c
Rhyme key = last stressed syllable + following consonants from phonology

In [5]:
def rhyme_key_from_phonology(phonology_json: str) -> str:
    """Derive rhyme key from phonology (last stressed vowel + following). Returns '' if unavailable."""
    if not phonology_json:
        return ""
    try:
        ph = json.loads(phonology_json)
    except Exception:
        return ""
    phones = []
    for p in ph:
        arp = p.get("arpabet")
        if isinstance(arp, list) and arp:
            phones.extend(arp[0].split() if isinstance(arp[0], str) else arp[0])
        elif isinstance(arp, str):
            phones.extend(arp.split())
    if not phones:
        return ""
    # Last stressed vowel onwards (ARPAbet: 0/1/2 = stress)
    for i in range(len(phones) - 1, -1, -1):
        p = phones[i]
        if len(p) > 1 and p[-1] in "12":
            return " ".join(phones[i:])
    return " ".join(phones[-3:]) if len(phones) >= 3 else " ".join(phones)

def build_rhyme_pairs(conn: sqlite3.Connection, poem_ids: list) -> list:
    out = []
    for line in iter_lines(conn, poem_ids):
        text = (line.get("normalized") or "").strip()
        rg = line.get("rhyme_group") or ""
        key = rhyme_key_from_phonology(line.get("phonology") or "")
        if not text:
            continue
        target = key if key else (rg if rg and rg != "-" else "none")
        out.append({"input": text, "target": target})
    return out

conn = sqlite3.connect(DB_PATH)
rhyme_train = build_rhyme_pairs(conn, train_ids)
rhyme_dev = build_rhyme_pairs(conn, dev_ids)
rhyme_test = build_rhyme_pairs(conn, test_ids)
conn.close()
print(f"Rhyme-only: train={len(rhyme_train)}, dev={len(rhyme_dev)}, test={len(rhyme_test)}")
if rhyme_train:
    ex = next((r for r in rhyme_train if r["target"] != "none"), rhyme_train[0])
    print(f"  Example: input={ex['input'][:50]}... target={ex['target']}")

Rhyme-only: train=206956, dev=42831, test=43535
  Example: input=Tired Nature's sweet restorer, balmy Sleep!... target=IY1 P


#### Combined structured supervision (continuation)

Same **input** as natural_text prior line/lines in the poem or start. Output = bundled **stress pattern** and **meter_type** label plus rhyme / end-stopped / caesura for the **next** line stress:…|meter_type:…|rhyme:…|end:…|caesura:…. Each row also stores next_line that line's surface text for strict-eval alignment in run_prompt_eval.py.

In [6]:
def build_combined_pairs(
    conn: sqlite3.Connection, poem_ids: list, context_lines: int = 1
) -> list:
    out = []
    lines = list(iter_lines(conn, poem_ids))
    i = 0
    while i < len(lines):
        line = lines[i]
        text = (line.get("normalized") or "").strip()
        if not text:
            i += 1
            continue
        inp = continuation_context_input(lines, i, context_lines, line["poem_id"])
        stress_raw = (line.get("stress") or "").strip()
        stress_pm = stress_to_plus_minus(stress_raw)
        meter_type = (line.get("meter_type") or "").strip() or "unknown"
        rg = line.get("rhyme_group") or "-"
        key = rhyme_key_from_phonology(line.get("phonology") or "")
        rhyme_tok = key if key else (rg if rg != "-" else "none")
        end = 1 if line.get("end_stopped") else 0
        ces = line.get("caesura")
        ces_tok = str(ces) if ces is not None else "-"
        target = (
            f"stress:{stress_pm or '-'}|meter_type:{meter_type}|"
            f"rhyme:{rhyme_tok}|end:{end}|caesura:{ces_tok}"
        )
        out.append({"input": inp, "target": target, "next_line": text})
        i += 1
    return out

conn = sqlite3.connect(DB_PATH)
comb_train = build_combined_pairs(conn, train_ids)
comb_dev = build_combined_pairs(conn, dev_ids)
comb_test = build_combined_pairs(conn, test_ids)
conn.close()
print(f"Combined: train={len(comb_train)}, dev={len(comb_dev)}, test={len(comb_test)}")
if comb_train:
    print(f"  Example: {comb_train[0]}")

Combined: train=206956, dev=42831, test=43535
  Example: {'input': '[start]', 'target': 'stress:+-+-+-+-+-+|meter_type:iambic Pentameter|rhyme:IY1 P|end:1|caesura:3', 'next_line': "Tired Nature's sweet restorer, balmy Sleep!"}


#### Export to JSON

In [7]:
for name, train, dev, test in [
    ("natural_text", nat_train, nat_dev, nat_test),
    ("meter_only", meter_train, meter_dev, meter_test),
    ("rhyme_only", rhyme_train, rhyme_dev, rhyme_test),
    ("combined", comb_train, comb_dev, comb_test),
]:
    out_path = OUT_DIR / name
    out_path.mkdir(parents=True, exist_ok=True)
    for split, data in [("train", train), ("dev", dev), ("test", test)]:
        with open(out_path / f"{split}.json", "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Wrote {out_path} (train={len(train)}, dev={len(dev)}, test={len(test)})")

print(f"\nAll training data saved to {OUT_DIR}")

Wrote /Users/kemiomoniyi/Senior Thesis/output/training_data/natural_text (train=206956, dev=42831, test=43535)
Wrote /Users/kemiomoniyi/Senior Thesis/output/training_data/meter_only (train=175767, dev=36209, test=36181)
Wrote /Users/kemiomoniyi/Senior Thesis/output/training_data/rhyme_only (train=206956, dev=42831, test=43535)
Wrote /Users/kemiomoniyi/Senior Thesis/output/training_data/combined (train=206956, dev=42831, test=43535)

All training data saved to /Users/kemiomoniyi/Senior Thesis/output/training_data
